<table><tr>
<td><b>Sapienza University of Rome</b><br>PhD Soft Skills 2026</td>
</tr></table>

# Notebook 3 — Look inside the language model
**20 minutes.** Tokens, probabilities, temperature — and a hallucination you will produce on purpose.


---
### How to use this notebook

1. Each numbered section gives you **a prompt**. Copy it.
2. Click the **empty code cell** underneath it.
3. Press `Ctrl`+`Shift`+`Enter` (or click **Generate**) to open Colab's AI, paste the prompt, and let it write the code.
4. **Look at what it wrote** — even if you don't understand it — then press ▶.
5. If it fails, don't fix it by hand. Paste the whole red error message back to the AI.

Every section also has a **Reference** cell with working code already in it. If you would rather watch than type, just run that instead. You will lose nothing.

> **Before you start: click `Copy to Drive` in the toolbar.** This notebook opened straight from GitHub, which means it is read-only — you can run it, but nothing you do will be saved. One click makes it yours to keep.

> You are never expected to write code. You are expected to say clearly what you want, and to check what you get.
---

> **A note on what runs here.** Sections 1–4 run entirely inside this notebook, with no model and no account needed. Section 5 uses the **Colab AI chat panel** (the sidebar), not a code cell — because that is where a real language model lives.

In [ ]:
# SETUP — run this once. It unpacks the course corpus into this notebook.
# Nothing is downloaded and nothing is installed.
import base64, gzip, io, tarfile, pathlib

_BLOB = (
    "..."
)

_tar = tarfile.open(fileobj=io.BytesIO(gzip.decompress(base64.b64decode(_BLOB))))
try:
    _tar.extractall('.', filter='data')   # Python 3.12+
except TypeError:
    _tar.extractall('.')                  # older Python
print('Ready. Files now available:')
for p in sorted(pathlib.Path('.').glob('*.csv')):
    print('  ', p, f'({p.stat().st_size//1024} KB)')
print('   archive/ :', len(list(pathlib.Path('archive').glob('*.txt'))), 'text files')

---
## 1 · Tokens: what the model actually sees

The model never sees words. It sees **tokens** — fragments from a fixed vocabulary. Let's look at real ones.

### 1 · Split text into real tokens

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Install and use the tiktoken library. Get the 'cl100k_base' encoding
and keep it in a variable called `enc`.
Take this sentence: 'The archivist catalogued an unremarkable manuscript.'
Split it into tokens and print each token with its number, one per line,
so I can see exactly where the splits fall.
Then print how many tokens it used and how many words it had.
```

> `tiktoken` is small and fast and pulls in no deep-learning framework. It is the actual tokenizer used by several production models. The variable name is pinned because later sections reuse it.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
!pip -q install tiktoken
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')
s = 'The archivist catalogued an unremarkable manuscript.'
ids = enc.encode(s)

print(f'{len(s.split())} words  ->  {len(ids)} tokens\n')
for i in ids:
    print(f'  {i:>7}  |{enc.decode([i])}|')

### 2 · The language tax

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Using the tiktoken 'cl100k_base' encoding, compare how many tokens are
needed for the same meaning in three languages. Use these sentences,
and mind the apostrophe in the Italian one - use double quotes around it:
  English: "The archivist catalogued an unremarkable manuscript."
  Italian: "L'archivista ha catalogato un manoscritto non degno di nota."
  Greek:   "Ο αρχειοφύλακας κατέγραψε ένα ασήμαντο χειρόγραφο."
Show the token count for each as a bar chart, and print the
tokens-per-character ratio for each.
```

> The quoting note is there for a reason: `'L'archivista'` ends the string at the apostrophe and produces a syntax error. If that happens to you, paste the red message back to the AI — it is a good first taste of the debugging loop.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import matplotlib.pyplot as plt

samples = {
    "English": "The archivist catalogued an unremarkable manuscript.",
    "Italian": "L'archivista ha catalogato un manoscritto non degno di nota.",
    "Greek":   "Ο αρχειοφύλακας κατέγραψε ένα ασήμαντο χειρόγραφο.",
}
counts = {k: len(enc.encode(v)) for k, v in samples.items()}
for k, v in samples.items():
    print(f'{k:9s} {counts[k]:3d} tokens for {len(v):3d} characters ({counts[k]/len(v):.2f} tokens per character)')

print()
print(f'Greek costs {counts["Greek"]/counts["English"]:.1f}x as many tokens as English for the same sentence.')

plt.figure(figsize=(6, 3.2))
plt.bar(counts.keys(), counts.values(), color=['#8E2436', '#B84A5C', '#A87A18'])
plt.ylabel('tokens for the same sentence'); plt.tight_layout(); plt.show()

---
## 2 · Next-token prediction, made visible

A language model does exactly one thing: given the text so far, it produces a **probability for every possible next token**.

We cannot open a frontier model here, so we will build a tiny one from our own letters — a *word-level* predictor that looks at the previous word and asks what usually comes next. It is enormously simpler than a real model, but what it produces has the same shape: a ranked list of candidates with probabilities.

### 3 · Build a tiny next-word predictor

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Using the text column of letters.csv, count which words follow which
single word. Store it as a dictionary called `nxt` mapping each word to
a Counter of the words that follow it. Lowercase everything, and treat
. , ; : as separate words.

Then write a function distribution(word) that returns the most likely
next words with their probabilities. If the word never appears, print a
clear message saying so rather than failing.

Show me the top 10 candidates after the word 'the', with their
probabilities, as a horizontal bar chart.
Use only plain Python, pandas and matplotlib — no neural networks.
```

> This is a *bigram model*. Real language models replaced this approach long ago, but the output — a ranked distribution over what comes next — is exactly what a modern model produces too, just over 100,000 fragments instead of a few hundred words.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import pandas as pd, collections, re, matplotlib.pyplot as plt

df = pd.read_csv('letters.csv')
words = []
for t in df['text']:
    words += re.findall(r"[\w']+|[.,;:]", t.lower())

nxt = collections.defaultdict(collections.Counter)
for a, b in zip(words, words[1:]):
    nxt[a][b] += 1

def distribution(word, k=10):
    counter = nxt.get(word.lower())
    if not counter:
        print(f'The word {word!r} never appears in these letters — try another.')
        return []
    total = sum(counter.values())
    return [(w, n / total) for w, n in counter.most_common(k)]

WORD = 'the'
top = distribution(WORD)
print(f"{sum(nxt[WORD].values())} times {WORD!r} appears, "
      f"followed by {len(nxt[WORD])} different words\n")
for w, p in top:
    print(f'  {p:6.1%}  {w}')

plt.figure(figsize=(7, 4))
plt.barh([w for w, _ in top][::-1], [p for _, p in top][::-1], color='#8E2436')
plt.xlabel('probability of being the next word')
plt.title(f"what comes after '{WORD}'")
plt.tight_layout(); plt.show()

---
## 3 · Temperature: the same model, different dice

Having a ranked list is not the same as choosing from it. **Temperature** controls how adventurous that choice is.

- Low temperature → almost always take the most likely word. Repetitive, predictable.
- High temperature → often take unlikely words. Varied, then incoherent.

### 4 · Generate at different temperatures

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Using the `nxt` predictor from the previous cell, write a function that
generates 22 words starting from the word 'i', choosing each next word
randomly in proportion to its count raised to the power 1/temperature.

Generate 2 samples at temperature 0.2, 2 at temperature 1.0, and 2 at
temperature 2.0. Print them grouped under clear headings and wrapped
so I can read them.
```

> Two things you will notice. The letters are **bilingual**, so English and Italian run together in the output — that is the corpus, not a bug. And read the three groups side by side: low temperature gets stuck in loops (*the most humble and the most humble and…*); high temperature wanders off the subject entirely. This is the same dial you set on any commercial model — and it is why the same question can give you different answers.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import random, numpy as np, textwrap

def generate(start='i', n=22, temperature=1.0, seed=0):
    rng = random.Random(seed)
    out = [start]
    for _ in range(n):
        counter = nxt.get(out[-1])
        if not counter:
            break
        cand = list(counter)
        w = np.array([counter[c] for c in cand], dtype=float)
        w = w ** (1.0 / max(temperature, 0.01))   # temperature reshapes the odds
        out.append(rng.choices(cand, weights=w / w.sum())[0])
    return ' '.join(out)

for t in (0.2, 1.0, 2.0):
    print(f'=== temperature {t} ' + '=' * 40)
    for s in range(2):
        print(textwrap.fill(generate(temperature=t, seed=s), 88))
        print()

---
## 4 · The tokenizer's revenge

Now a famous failure that follows directly from section 1. Ask a language model how many times the letter **r** appears in *strawberry* and it has historically got it wrong.

It never saw letters. It saw the fragments `str`, `aw`, `berry`. Counting characters is not an operation available to it.

### 5 · Count it properly

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Count how many times each letter appears in the word 'strawberry',
and show how the tiktoken encoding `enc` splits that same word.
Print the two results side by side.
```

> Then ask the Colab AI chat panel the same question in words: *how many times does the letter r appear in strawberry?* Compare its answer to the count you just computed. Recent models often get this right — if yours does, ask it for a longer or rarer word and try again.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import collections

word = 'strawberry'
print('the truth, counted directly:')
for ch, n in sorted(collections.Counter(word).items()):
    print(f'   {ch}: {n}')

print('\nwhat a model actually receives:')
for i in enc.encode(word):
    print(f'   |{enc.decode([i])}|')
print('\nNo letters. Just chunks. Counting characters is not an operation it can do.')

---
## 5 · Make it hallucinate — on purpose

**This section uses the Colab AI chat panel** (the sidebar), not a code cell.

You are going to ask for something the model almost certainly does not know, and watch it produce something that looks exactly like knowledge.

**Before you paste anything: replace the bracketed part with a real topic of your own.** The narrower and more specialised, the sharper the result. A general topic returns real citations and the lesson is lost.

### 6 · Ask for citations in your own narrow specialism

**Paste this into the Colab AI chat panel (not a code cell)**:

```text
Give me five peer-reviewed articles about TOPIC, with the authors, the
title, the journal, the year, and the DOI for each.

  ^^^^^ replace TOPIC with the narrowest subject in your own research
        that you can state in one line, then delete this note
```

> If you paste it without replacing TOPIC, the model will ask you what topic you mean — which wastes a turn but does no harm.

### Now verify every single one

Search for each title. Search for each DOI. Check that each author exists and works in that field.

**Write down how many of the five were real.** Keep that number — it is for you, not for anyone else.

Then ask this follow-up in the same chat:

### 7 · Ask it to mark its own work

**Paste this into the Colab AI chat panel (not a code cell)**:

```text
For each of the five citations you just gave me, tell me how confident you
are that it exists, and mark any that you may have constructed rather than
recalled. Be blunt. Do not defend your earlier answer.
```

> Sometimes this works remarkably well. Sometimes it defends fiction with complete confidence. **The fact that you cannot tell which in advance is the entire point.** Self-checking is worth a prompt. It is never verification.

---
## What you just saw

| | |
|---|---|
| **Tokens** | The model sees fragments, not words or letters. Non-English text costs far more. |
| **Probabilities** | Its only output is a ranked distribution over what comes next. |
| **Temperature** | Choosing from that distribution is a dice roll you control. |
| **Hallucination** | A plausible citation and a real one are indistinguishable to that machinery. |

One honest caveat about our tiny predictor: it looks only at the **previous word**, so it forgets everything before that and wanders off the subject. A real model looks at thousands of tokens at once. That difference — and only that difference — is what Module 1 spent the morning on.

> **The rule that follows:** never cite anything an AI gave you that you have not personally opened. Not the abstract — the paper.